# Fit Neural Templates To Saved Template Checkpoints

This notebook answers a narrower question than survey training: given a saved `TemplateModel` checkpoint in `../checkpoints-templates/`, how well can the current `NeuralTemplateModel` parameterization reproduce those rest-frame templates directly?

It will:

- load one template checkpoint,
- fit neural templates directly to the saved template array,
- plot overlays and residuals,
- export a loadable neural checkpoint to `../checkpoints-neural-fit/`.


In [ ]:
import os
import sys
from pathlib import Path

# Default to CPU so the notebook runs cleanly on machines without a visible GPU.
os.environ.setdefault('JAX_PLATFORMS', 'cpu')

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, '../py')

from neural_template_fit import (
    build_template_array,
    export_fitted_neural_checkpoint,
    fit_neural_to_template_checkpoint,
    latest_checkpoint,
    load_wave_obs,
)
from neural_template_model import NeuralTemplateModel
from template_model import TemplateModel


In [ ]:
CHECKPOINT_DIR = Path('../checkpoints-templates')
SPECTRA_DIR = '../spectra'
OUTPUT_DIR = Path('../checkpoints-neural-fit')

# Default: latest template checkpoint. Replace with any specific file if needed.
TEMPLATE_CKPT = Path(latest_checkpoint(str(CHECKPOINT_DIR)))

MLP_HIDDEN = 64
LR = 3e-2
N_STEPS = 3000
SEED = 0

available = sorted(CHECKPOINT_DIR.glob('checkpoint_epoch*.npz'))
print('Using checkpoint:', TEMPLATE_CKPT)
print('Available checkpoints:', len(available))
print('First:', available[0].name)
print('Last: ', available[-1].name)


In [ ]:
wave_obs = load_wave_obs(SPECTRA_DIR)
template_model, template_params, template_epoch = TemplateModel.load_checkpoint(str(TEMPLATE_CKPT), wave_obs)
target_T = build_template_array(template_model, template_params)

neural_model, neural_params, fit_metrics = fit_neural_to_template_checkpoint(
    str(TEMPLATE_CKPT),
    wave_obs,
    mlp_hidden=MLP_HIDDEN,
    lr=LR,
    n_steps=N_STEPS,
    seed=SEED,
)
fit_T = build_template_array(neural_model, neural_params)
t_wave = np.array(template_model.t_wave)

print(f'Template checkpoint epoch: {template_epoch}')
print(f'Nt={template_model.Nt}, Nft={template_model.Nft}, z=[{template_model.zmin:.2f}, {template_model.zmax:.2f}]')
print(f"relative_mse={fit_metrics['relative_mse']:.6f}  rmse={fit_metrics['rmse']:.6f}  max_abs_err={fit_metrics['max_abs_err']:.6f}")


In [ ]:
line_sigma = np.array(np.log1p(np.exp(np.array(neural_params['line_sigma_raw'])))) + 0.1
print('Per-template metrics:')
for item in fit_metrics['per_template']:
    k = int(item['template'])
    print(
        f"  T{k}: corr={item['corr']:.4f}  rmse={item['rmse']:.6f}  max_abs_err={item['max_abs_err']:.6f}"
    )
print(f'Line sigma range: {line_sigma.min():.3f} .. {line_sigma.max():.3f} Angstrom')


In [ ]:
fig, axes = plt.subplots(template_model.Nt, 1, figsize=(13, 3.2 * template_model.Nt), sharex=True)
axes = np.atleast_1d(axes)

for k, ax in enumerate(axes):
    per_template = fit_metrics['per_template'][k]
    ax.plot(t_wave, target_T[k], color='black', lw=1.2, label='Template checkpoint')
    ax.plot(t_wave, fit_T[k], color='tab:orange', lw=1.0, alpha=0.9, label='Best-fit neural template')
    ax.set_ylabel(f'T{k}')
    ax.grid(alpha=0.2)
    ax.legend(loc='upper right')
    ax.set_title(
        f"Template {k}: corr={per_template['corr']:.4f}, rmse={per_template['rmse']:.5f}, max|err|={per_template['max_abs_err']:.5f}"
    )

axes[-1].set_xlabel('Rest-frame wavelength [Angstrom]')
fig.suptitle('Saved template-model templates vs direct best-fit neural templates', fontsize=13)
fig.tight_layout()


In [ ]:
residual = fit_T - target_T
fig, axes = plt.subplots(template_model.Nt, 1, figsize=(13, 2.6 * template_model.Nt), sharex=True)
axes = np.atleast_1d(axes)

for k, ax in enumerate(axes):
    ax.plot(t_wave, residual[k], color='tab:red', lw=0.9)
    ax.axhline(0.0, color='black', lw=0.8, alpha=0.6)
    ax.set_ylabel(f'resid T{k}')
    ax.grid(alpha=0.2)

axes[-1].set_xlabel('Rest-frame wavelength [Angstrom]')
fig.suptitle('Neural fit residuals (fit - target)', fontsize=13)
fig.tight_layout()


In [ ]:
print('Largest residual locations per template:')
for k in range(template_model.Nt):
    err = np.abs(residual[k])
    top = np.argsort(err)[-8:]
    print(f'  Template {k}')
    for idx in sorted(top):
        print(
            f"    wave={t_wave[idx]:7.1f}  target={target_T[k, idx]:+8.4f}  fit={fit_T[k, idx]:+8.4f}  abs_err={err[idx]:.4f}"
        )


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / TEMPLATE_CKPT.name.replace('.npz', '_neural_fit.npz')

export_fitted_neural_checkpoint(
    str(output_path),
    neural_model,
    neural_params,
    epoch=0,
    source_checkpoint=str(TEMPLATE_CKPT),
    fit_metrics=fit_metrics,
)

print('Saved:', output_path)


In [ ]:
reloaded_model, reloaded_params, reload_epoch = NeuralTemplateModel.load_checkpoint(str(output_path), wave_obs)
reloaded_T = build_template_array(reloaded_model, reloaded_params)
reload_delta = np.max(np.abs(reloaded_T - fit_T))

print(f'Reloaded checkpoint epoch={reload_epoch}')
print(f'Max |reload - fit| = {reload_delta:.6e}')
